# WildFusion: Animal Re-Identification Pipeline
AnimalCLEF 2025 Winner Architecture Overview

This notebook implements the WildFusion pipeline for animal re-identification. It addresses sensitivity to pose and occlusions by fusing global descriptors with local keypoint matching.

### Technical Requirements Implemented:
*   **Global Extractor**: MegaDescriptor-L-384 (via timm) for global distance computation.
*   **Local Matching**: ALIKED (detector) and LightGlue (matcher) for local inlier count.
*   **Score Fusion & Calibration**: Isotonic Regression to map raw scores to calibrated probability $P(match) \in [0, 1]$.
*   **Novelty Filtering**: Threshold-based separation for known vs. new individuals.
*   **Efficiency**: Global-First filter for top-K retrieval before local matching.


In [ ]:
# Setup dependencies (Uncomment to install if not present)
# !pip install kornia lightglue timm scikit-learn torch torchvision pillow requests huggingface_hub


In [ ]:
import torch
import torchvision.transforms as T
import timm
from PIL import Image
from sklearn.isotonic import IsotonicRegression
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Using lightglue and kornia for local matching
from lightglue import LightGlue, ALIKED
from lightglue.utils import load_image, rbd


### Model Loading (Lazy)
We define lazy-loading for the MegaDescriptor and LightGlue to optimize memory footprint.


In [ ]:
class ModelLoader:
    def __init__(self):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self._global_model = None
        self._local_extractor = None
        self._local_matcher = None
        
        # Transformations for global descriptor (timm based)
        self.global_transforms = T.Compose([
            T.Resize((384, 384)),
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

    @property
    def global_model(self):
        if self._global_model is None:
            print("Loading MegaDescriptor-L-384...")
            # Load MegaDescriptor-L-384 pre-trained on wildlife datasets
            self._global_model = timm.create_model('hf_hub:BVRA/MegaDescriptor-L-384', pretrained=True, num_classes=0)
            self._global_model = self._global_model.to(self.device).eval()
        return self._global_model

    @property
    def local_extractor(self):
        if self._local_extractor is None:
            print("Loading ALIKED extractor...")
            self._local_extractor = ALIKED(max_num_keypoints=2048).eval().to(self.device)
        return self._local_extractor

    @property
    def local_matcher(self):
        if self._local_matcher is None:
            print("Loading LightGlue matcher...")
            self._local_matcher = LightGlue(features='aliked').eval().to(self.device)
        return self._local_matcher

models = ModelLoader()


### Inference Pipeline
The `extract_global_feature` computes the MegaDescriptor. The `get_local_matches` computes the number of inliers using LightGlue. The `match_pair` function combines both.


In [ ]:
def extract_global_feature(img_path):
    img = Image.open(img_path).convert('RGB')
    tensor = models.global_transforms(img).unsqueeze(0).to(models.device)
    with torch.no_grad():
        feat = models.global_model(tensor)
        feat = torch.nn.functional.normalize(feat, p=2, dim=1)
    return feat

def get_local_matches(img_path1, img_path2):
    """
    Extracts local features and matches them using LightGlue.
    Returns the number of inlier matches.
    """
    img1 = load_image(img_path1).to(models.device)
    img2 = load_image(img_path2).to(models.device)
    
    # Extract features with ALIKED
    feats1 = models.local_extractor.extract(img1)
    feats2 = models.local_extractor.extract(img2)
    
    # Match features with LightGlue
    matches01 = models.local_matcher({"image0": feats1, "image1": feats2})
    
    # Remove batch dimension to get a simple dict of 2D tensors
    matches01_unbatched = rbd(matches01)
    
    # Get number of inliers (matches)
    matches = matches01_unbatched['matches']  # Shape is (M, 2) where M is the number of matching point pairs
    local_score = matches.shape[0] if matches is not None else 0
    
    return local_score

def match_pair(img_path1, img_path2):
    """
    Returns both global distance and local match count for two images.
    Global Distance: 1 - cosine_similarity
    Local Score: Number of inlier matches
    """
    # Global Extraction
    feat1 = extract_global_feature(img_path1)
    feat2 = extract_global_feature(img_path2)
    
    cosine_sim = torch.nn.functional.cosine_similarity(feat1, feat2).item()
    global_dist = 1.0 - cosine_sim
    
    # Local Match Count
    local_score = get_local_matches(img_path1, img_path2)
    
    return global_dist, local_score


### Score Fusion & Calibration Training
Using `IsotonicRegression` from sklearn to map the raw global distance and local score into a calibrated probability $P(match) \in [0, 1]$.
We combine the two scores into a single metric for the regression. E.g., `combined_score = (1 - global_dist) * local_score`.


In [ ]:
class ScoreCalibrator:
    def __init__(self):
        self.iso_reg = IsotonicRegression(out_of_bounds='clip')
        
    def _combine_scores(self, global_dist, local_score):
        # We want higher score for better matches. 
        # Lower global_dist is better, higher local_score is better.
        similarity = 1.0 - global_dist
        # Add small epsilon to similarity to prevent zeroing out good local scores if similarity is slightly off
        return max(0, similarity) * np.log1p(local_score)

    def fit(self, global_dists, local_scores, labels):
        """
        global_dists: list or array of global distances
        local_scores: list or array of local match counts
        labels: 1 for match, 0 for non-match
        """
        combined = [self._combine_scores(g, l) for g, l in zip(global_dists, local_scores)]
        self.iso_reg.fit(combined, labels)
        print("Calibration complete.")
        
    def predict_proba(self, global_dist, local_score):
        combined = self._combine_scores(global_dist, local_score)
        return self.iso_reg.predict([combined])[0]

# Mock-up Calibration Training
print("Running Mock Calibration...")
mock_g_dists = [0.1, 0.8, 0.2, 0.9, 0.15, 0.85]
mock_l_scores = [150, 5, 120, 2, 80, 0]
mock_labels = [1, 0, 1, 0, 1, 0] # 1 = match, 0 = non-match

calibrator = ScoreCalibrator()
calibrator.fit(mock_g_dists, mock_l_scores, mock_labels)

# Test prediction
p_match = calibrator.predict_proba(0.12, 100)
print(f"Test Probability of Match: {p_match:.4f}")


### Search & Identification (WildFusionSystem)
Maintains a database of known animals and performs Top-K retrieval using global descriptors, followed by local verification to save compute time.
Threshold-based novelty detection is used to identify new individuals.


In [ ]:
class WildFusionSystem:
    def __init__(self, calibrator, novelty_threshold=0.5, top_k=5):
        self.calibrator = calibrator
        self.novelty_threshold = novelty_threshold
        self.top_k = top_k
        self.database = {} # id -> list of image paths
        self.db_features = [] # List of tuples: (id, feature_tensor, img_path)
        
    def add_known_individual(self, individual_id, img_path):
        if individual_id not in self.database:
            self.database[individual_id] = []
        self.database[individual_id].append(img_path)
        
        # Precompute global feature
        feat = extract_global_feature(img_path)
        self.db_features.append((individual_id, feat, img_path))
        
    def identify(self, query_img_path):
        """
        Global-First filter: 
        1. Compare query against all global features in DB.
        2. Retrieve Top-K candidates based on global distance.
        3. Run local matching ONLY on Top-K candidates.
        4. Apply calibrated fusion and novelty threshold.
        """
        if not self.db_features:
            return "New Individual (Empty DB)", 0.0

        # Extract global feature once for the query
        query_feat = extract_global_feature(query_img_path)
        
        # 1. Global Search
        global_scores = []
        for db_id, db_feat, db_img_path in self.db_features:
            cosine_sim = torch.nn.functional.cosine_similarity(query_feat, db_feat).item()
            global_dist = 1.0 - cosine_sim
            global_scores.append((global_dist, db_id, db_img_path))
            
        # 2. Retrieve Top-K candidates
        global_scores.sort(key=lambda x: x[0]) # Ascending distance (lower is better)
        top_k_candidates = global_scores[:self.top_k]
        
        # 3. Local Verification on Top-K
        best_match_id = None
        best_p_match = 0.0
        
        for g_dist, db_id, db_img_path in top_k_candidates:
            # We already have g_dist, so ONLY compute local score (no redundant ViT forward passes)
            l_score = get_local_matches(query_img_path, db_img_path)
            
            # 4. Calibrated Fusion
            p_match = self.calibrator.predict_proba(g_dist, l_score)
            
            if p_match > best_p_match:
                best_p_match = p_match
                best_match_id = db_id
                
        # Novelty Filtering
        if best_p_match >= self.novelty_threshold:
            return f"Known Individual: {best_match_id}", best_p_match
        else:
            return "New Individual", best_p_match
